# Notebook 01 – EDA & Tiền xử lý

**Đề tài 5: Dự báo Thời tiết**  
**Dataset:** Kaggle Weather History (96,453 × 12, dữ liệu theo giờ 2006–2016)  

Notebook này thực hiện:
1. Load & kiểm tra schema dữ liệu
2. Thống kê mô tả & phát hiện vấn đề (missing, outlier, imbalance)
3. Tiền xử lý: làm sạch, encoding, tạo nhãn WeatherType
4. EDA với ≥6 biểu đồ + diễn giải tự động từ dữ liệu
5. So sánh thống kê trước–sau tiền xử lý

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import matplotlib
matplotlib.use('Agg')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, warnings
warnings.filterwarnings('ignore')
from src.data.loader import load_config, load_raw_data, validate_schema
from src.data.cleaner import DataCleaner
from src.visualization import plots
from src.evaluation.report import save_table
cfg = load_config('../configs/params.yaml')
plots.FIG_DIR = '../outputs/figures'
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

## 1. Load dữ liệu & kiểm tra schema

In [2]:
df_raw = load_raw_data('../' + cfg['paths']['raw_data'])
validate_schema(df_raw)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

[loader] Loaded 96,453 rows × 12 cols from ../data/raw/weatherHistory.csv
[loader] Schema OK.
Shape: (96453, 12)


,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-03-31 22:00:00+00:00,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-03-31 23:00:00+00:00,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 00:00:00+00:00,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.


## 2. Kiểm tra Missing, Kiểu dữ liệu & Thống kê mô tả

In [3]:
print('--- Kiểu dữ liệu ---')
print(df_raw.dtypes)
print('\n--- Missing Values ---')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Pct(%)': missing_pct})
print(missing_df[missing_df['Count'] > 0].to_string())
print(f'\nTổng missing: {missing.sum()} ({missing.sum()/len(df_raw)*100:.2f}%)')

--- Kiểu dữ liệu ---
Formatted Date              datetime64[ns, UTC]
Summary                                  object
Precip Type                              object
Temperature (C)                         float64
Apparent Temperature (C)                float64
Humidity                                float64
Wind Speed (km/h)                       float64
Wind Bearing (degrees)                  float64
Visibility (km)                         float64
Loud Cover                              float64
Pressure (millibars)                    float64
Daily Summary                            object
dtype: object

--- Missing Values ---
             Count  Pct(%)
Precip Type    517    0.54

Tổng missing: 517 (0.54%)


In [4]:
df_raw.describe().round(3)

,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars)
count,96453.000,96453.000,96453.000,96453.000,96453.000,96453.000,96453.0,96453.000
mean,11.933,10.855,0.735,10.811,187.509,10.347,0.0,1003.236
std,9.552,10.697,0.195,6.914,107.383,4.192,0.0,116.970
min,-21.822,-27.717,0.000,0.000,0.000,0.000,0.0,0.000
25%,4.689,2.311,0.600,5.828,116.000,8.340,0.0,1011.900
50%,12.000,12.000,0.780,9.966,180.000,10.046,0.0,1016.450
75%,18.839,18.839,0.890,14.136,290.000,14.812,0.0,1021.090
max,39.906,39.344,1.000,63.853,359.000,16.100,0.0,1046.380


## 3. Tiền xử lý (Preprocessing)

**Quy trình:**
- Bỏ cột `Loud Cover` (toàn bộ = 0) và `Daily Summary` (text thô)
- Fill missing `Precip Type` = "unknown"
- Loại outlier: `Pressure ≤ 800 mbar`
- Tạo nhãn `WeatherType` từ `Summary` (5 nhóm: Clear/Cloudy/Foggy/Windy/Rainy)
- Tạo đặc trưng thời gian: Hour, Month, DayOfWeek, Season
- Encode `PrecipType`

In [5]:
cleaner = DataCleaner(cfg)
df = cleaner.clean(df_raw)

[cleaner] Input shape: (96453, 12)
[cleaner] Dropped 0 rows with missing numeric values
[cleaner] Dropped 1288 rows with Pressure <= 800 (outlier)
[cleaner] Output shape: (95150, 16)
[cleaner] WeatherType distribution:
WeatherType
Cloudy    76170
Clear     10196
Foggy      7085
Windy      1587
Rainy       112
Name: count, dtype: int64


### 3.1 Thống kê trước–sau tiền xử lý

So sánh các chỉ số thống kê trước và sau khi làm sạch:

In [6]:
stats = cleaner.get_stats(df_raw, df)
print(stats.to_string(index=False))
save_table(stats, 'preprocessing_stats', '../outputs/tables')

# Tóm tắt thay đổi
print(f'\nSố dòng trước: {len(df_raw):,} → sau: {len(df):,} (giảm {len(df_raw)-len(df):,})')
print(f'Số cột trước: {df_raw.shape[1]} → sau: {df.shape[1]}')
print(f'Missing sau xử lý: {df.isnull().sum().sum()}')

              Column  Before_mean  After_mean  Before_missing  After_missing  Before_rows  After_rows
     Temperature (C)       11.933      11.948               0              0        96453       95150
            Humidity        0.735       0.735               0              0        96453       95150
   Wind Speed (km/h)       10.811      10.782               0              0        96453       95150
     Visibility (km)       10.347      10.382               0              0        96453       95150
Pressure (millibars)     1003.236    1016.814               0              0        96453       95150
[report] Saved table: ../outputs/tables\preprocessing_stats.csv

Số dòng trước: 96,453 → sau: 95,150 (giảm 1,303)
Số cột trước: 12 → sau: 16
Missing sau xử lý: 0


In [7]:
# Lưu processed data
os.makedirs('../data/processed', exist_ok=True)
df.to_parquet('../data/processed/weather_processed.parquet', index=False)
print(f'Saved processed data: {df.shape}')

Saved processed data: (95150, 16)


## 4. EDA – Khám phá dữ liệu

Tối thiểu 6 biểu đồ, mỗi biểu đồ kèm diễn giải **tự động từ dữ liệu**.

### Biểu đồ 1: Phân phối nhiệt độ

In [8]:
plots.plot_temperature_distribution(df)

temp = df['Temperature (C)']
print(f'Nhiệt độ: mean={temp.mean():.1f}°C, median={temp.median():.1f}°C, std={temp.std():.1f}°C')
print(f'  Khoảng Q1–Q3: {temp.quantile(0.25):.1f}°C – {temp.quantile(0.75):.1f}°C')
skew_val = temp.skew()
shape = 'lệch phải' if skew_val > 0.5 else 'lệch trái' if skew_val < -0.5 else 'xấp xỉ đối xứng'
print(f'  Skewness = {skew_val:.2f} → phân phối {shape}')
n_outlier_low = (temp < temp.quantile(0.01)).sum()
n_outlier_high = (temp > temp.quantile(0.99)).sum()
print(f'  Outlier (1% đuôi): thấp={n_outlier_low}, cao={n_outlier_high}')

[plots] Saved: ../outputs/figures\eda_temperature_dist.png
Nhiệt độ: mean=11.9°C, median=12.0°C, std=9.5°C
  Khoảng Q1–Q3: 4.7°C – 18.8°C
  Skewness = 0.09 → phân phối xấp xỉ đối xứng
  Outlier (1% đuôi): thấp=949, cao=937


### Biểu đồ 2: Phân bố loại thời tiết (target)

In [9]:
plots.plot_weather_type_counts(df, 'WeatherType')

vc = df['WeatherType'].value_counts()
total = len(df)
for wtype, count in vc.items():
    print(f'  {wtype}: {count:,} ({count/total*100:.1f}%)')
print(f'→ Lớp chiếm đa số: {vc.index[0]} ({vc.iloc[0]/total*100:.1f}%)')
print(f'→ Lớp thiểu số: {vc.index[-1]} ({vc.iloc[-1]/total*100:.1f}%)')
imbalance_ratio = vc.iloc[0] / vc.iloc[-1]
print(f'→ Tỷ lệ mất cân bằng: {imbalance_ratio:.1f}:1')

[plots] Saved: ../outputs/figures\eda_WeatherType_counts.png
  Cloudy: 76,170 (80.1%)
  Clear: 10,196 (10.7%)
  Foggy: 7,085 (7.4%)
  Windy: 1,587 (1.7%)
  Rainy: 112 (0.1%)
→ Lớp chiếm đa số: Cloudy (80.1%)
→ Lớp thiểu số: Rainy (0.1%)
→ Tỷ lệ mất cân bằng: 680.1:1


### Biểu đồ 3: Ma trận tương quan

In [10]:
num_cols = [c for c in ['Temperature (C)', 'Apparent Temperature (C)', 'Humidity',
            'Wind Speed (km/h)', 'Wind Bearing (degrees)', 'Visibility (km)',
            'Pressure (millibars)'] if c in df.columns]
plots.plot_correlation_heatmap(df, num_cols)

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
corr_pairs = corr.where(mask).stack().sort_values()
print('Tương quan dương cao nhất:')
for (c1, c2), val in corr_pairs.tail(3).items():
    print(f'  {c1} ↔ {c2}: {val:.3f}')
print('Tương quan âm mạnh nhất:')
for (c1, c2), val in corr_pairs.head(3).items():
    print(f'  {c1} ↔ {c2}: {val:.3f}')

[plots] Saved: ../outputs/figures\eda_correlation_heatmap.png
Tương quan dương cao nhất:
  Apparent Temperature (C) ↔ Visibility (km): 0.381
  Temperature (C) ↔ Visibility (km): 0.393
  Temperature (C) ↔ Apparent Temperature (C): 0.993
Tương quan âm mạnh nhất:
  Temperature (C) ↔ Humidity: -0.633
  Apparent Temperature (C) ↔ Humidity: -0.604
  Humidity ↔ Visibility (km): -0.371


### Biểu đồ 4: Nhiệt độ trung bình theo tháng

In [11]:
plots.plot_monthly_temperature(df)

monthly = df.groupby('Month')['Temperature (C)'].mean()
mn = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
print(f'Tháng nóng nhất: {mn[monthly.idxmax()]} = {monthly.max():.1f}°C')
print(f'Tháng lạnh nhất: {mn[monthly.idxmin()]} = {monthly.min():.1f}°C')
print(f'Biên độ nhiệt năm: {monthly.max() - monthly.min():.1f}°C')

[plots] Saved: ../outputs/figures\eda_monthly_temperature.png
Tháng nóng nhất: Jul = 22.9°C
Tháng lạnh nhất: Jan = 0.8°C
Biên độ nhiệt năm: 22.1°C


### Biểu đồ 5: Phân bố thời tiết theo mùa

In [12]:
plots.plot_season_weather(df)

ct = pd.crosstab(df['Season'], df['WeatherType'], normalize='index') * 100
print(ct.round(1).to_string())
print()
for season in ['Spring', 'Summer', 'Fall', 'Winter']:
    if season in ct.index:
        row = ct.loc[season]
        top = row.idxmax()
        print(f'  {season}: chiếm tỷ lệ cao nhất → {top} ({row[top]:.1f}%)')

[plots] Saved: ../outputs/figures\eda_season_weather.png
WeatherType  Clear  Cloudy  Foggy  Rainy  Windy
Season                                         
Fall          11.8    78.0    8.4    0.4    1.4
Spring         9.9    85.4    1.9    0.1    2.6
Summer        14.2    84.7    0.6    0.0    0.5
Winter         6.8    71.8   19.3    0.0    2.2

  Spring: chiếm tỷ lệ cao nhất → Cloudy (85.4%)
  Summer: chiếm tỷ lệ cao nhất → Cloudy (84.7%)
  Fall: chiếm tỷ lệ cao nhất → Cloudy (78.0%)
  Winter: chiếm tỷ lệ cao nhất → Cloudy (71.8%)


### Biểu đồ 6: Độ ẩm trung bình theo giờ

In [13]:
plots.plot_hourly_humidity(df)

hourly = df.groupby('Hour')['Humidity'].mean()
print(f'Giờ ẩm nhất: {hourly.idxmax()}h (Humidity={hourly.max():.3f})')
print(f'Giờ khô nhất: {hourly.idxmin()}h (Humidity={hourly.min():.3f})')
print(f'Biên độ ẩm ngày: {hourly.max() - hourly.min():.3f}')

[plots] Saved: ../outputs/figures\eda_hourly_humidity.png
Giờ ẩm nhất: 3h (Humidity=0.871)
Giờ khô nhất: 13h (Humidity=0.569)
Biên độ ẩm ngày: 0.302


## 5. Tóm tắt EDA

In [14]:
print('=== TÓM TẮT EDA ===')
print(f'Tổng bản ghi sau xử lý: {len(df):,}')
print(f'Khoảng thời gian: {df["Formatted Date"].min()} → {df["Formatted Date"].max()}')
print(f'Số loại thời tiết (target): {df["WeatherType"].nunique()} nhóm')
print(f'Missing sau xử lý: {df.isnull().sum().sum()}')
print()
print('Phân bố target WeatherType:')
for wtype, cnt in df['WeatherType'].value_counts().items():
    print(f'  {wtype}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

=== TÓM TẮT EDA ===
Tổng bản ghi sau xử lý: 95,150
Khoảng thời gian: 2005-12-31 23:00:00+00:00 → 2016-12-31 22:00:00+00:00
Số loại thời tiết (target): 5 nhóm
Missing sau xử lý: 0

Phân bố target WeatherType:
  Cloudy: 76,170 (80.1%)
  Clear: 10,196 (10.7%)
  Foggy: 7,085 (7.4%)
  Windy: 1,587 (1.7%)
  Rainy: 112 (0.1%)
